# Лабораторна робота 5: Реалізація SCD Type 2

Повільно змінювані виміри — відстеження історії даних у DWH.

---
**Мета:** реалізувати SCD Type 2, Type 1, Type 3 на реальних даних клієнтів.

In [ ]:
import sqlite3
import pandas as pd
from datetime import datetime, date

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row
print('SQLite connected')

---
## 1. SCD Type 1: Overwrite

Історія втрачається — тільки поточне значення.

In [ ]:
conn.executescript('''
    CREATE TABLE DimCustomer_T1 (
        customer_id INT PRIMARY KEY,
        name TEXT,
        city TEXT,
        loyalty_tier TEXT
    );

    INSERT INTO DimCustomer_T1 VALUES (1, 'Alice', 'Lviv', 'Silver');
    INSERT INTO DimCustomer_T1 VALUES (2, 'Bob', 'Kyiv', 'Gold');
''')

print('Before update:')
display(pd.read_sql('SELECT * FROM DimCustomer_T1', conn))

# Клієнт переїхав — старе місто втрачено
conn.execute("UPDATE DimCustomer_T1 SET city = 'Odesa' WHERE customer_id = 1")

print('After update (Type 1 — історію втрачено):')
display(pd.read_sql('SELECT * FROM DimCustomer_T1', conn))

---
## 2. SCD Type 2: Add new row (НАЙВАЖЛИВІШИЙ)

При зміні — новий рядок з датами effective/end.

In [ ]:
conn.executescript('''
    CREATE TABLE DimCustomer_SCD2 (
        customer_key INTEGER PRIMARY KEY AUTOINCREMENT,
        customer_id INT,
        name TEXT,
        city TEXT,
        loyalty_tier TEXT,
        effective_date DATE,
        end_date DATE,
        is_current BOOLEAN DEFAULT 1
    );

    -- Initial load (2022-01-01)
    INSERT INTO DimCustomer_SCD2 (customer_id, name, city, loyalty_tier, effective_date, end_date, is_current)
    VALUES 
        (1, 'Alice', 'Lviv', 'Silver', '2022-01-01', NULL, 1),
        (2, 'Bob', 'Kyiv', 'Gold', '2022-01-01', NULL, 1);
''')
print('Initial state:')
display(pd.read_sql('SELECT * FROM DimCustomer_SCD2', conn))

In [ ]:
# SCD Type 2 UPDATE (спрощена процедура)
# Alice переїжджає зі Львова до Києва (2024-06-01)

def scd2_update(customer_id, column, new_value, change_date, conn):
    # 1. Закриваємо поточний запис
    conn.execute('''
        UPDATE DimCustomer_SCD2 
        SET end_date = ?, is_current = 0
        WHERE customer_id = ? AND is_current = 1
    ''', (change_date, customer_id))
    
    # 2. Отримуємо попередні значення
    old = conn.execute('SELECT * FROM DimCustomer_SCD2 WHERE customer_key = (
        SELECT MAX(customer_key) FROM DimCustomer_SCD2 WHERE customer_id = ?
    )', (customer_id,)).fetchone()
    
    # 3. Створюємо новий запис
    if column == 'city':
        conn.execute('''
            INSERT INTO DimCustomer_SCD2 (customer_id, name, city, loyalty_tier, effective_date, end_date, is_current)
            VALUES (?, ?, ?, ?, ?, NULL, 1)
        ''', (customer_id, old['name'], new_value, old['loyalty_tier'], change_date))
    elif column == 'loyalty_tier':
        conn.execute('''
            INSERT INTO DimCustomer_SCD2 (customer_id, name, city, loyalty_tier, effective_date, end_date, is_current)
            VALUES (?, ?, ?, ?, ?, NULL, 1)
        ''', (customer_id, old['name'], old['city'], new_value, change_date))

scd2_update(1, 'city', 'Kyiv', '2024-06-01', conn)
print('After Alice moved to Kyiv (2024-06-01):')
display(pd.read_sql('SELECT * FROM DimCustomer_SCD2 ORDER BY customer_id, effective_date', conn))

In [ ]:
# Ще одна зміна: Alice стала Platinum (2024-12-01)
scd2_update(1, 'loyalty_tier', 'Platinum', '2024-12-01', conn)
print('After Alice became Platinum:')
display(pd.read_sql('SELECT * FROM DimCustomer_SCD2 ORDER BY customer_id, effective_date', conn))

In [ ]:
# Як виглядають дані для аналітики?
# 
# Щоб дізнатись яким був клієнт на момент замовлення:
print('Alice at different points in time:')
for dt in ['2023-01-15', '2024-07-01', '2025-01-01']:
    row = conn.execute('''
        SELECT customer_id, name, city, loyalty_tier
        FROM DimCustomer_SCD2
        WHERE customer_id = 1 
          AND effective_date <= ?
          AND (end_date IS NULL OR end_date > ?)
    ''', (dt, dt)).fetchone()
    print(f'  {dt}: {dict(row)}')

In [ ]:
# JOIN Fact з Dimension SCD2 (правильно!)
# Зазвичай у Fact зберігається customer_key, який посилається на конкретну версію
print('Як виглядає JOIN з FactSales:')
query = '''
    SELECT 
        f.order_date,
        c.name,
        c.city AS customer_city_at_time,
        c.loyalty_tier AS tier_at_time,
        f.total_amount
    FROM (
        SELECT '2023-01-15' AS order_date, 1 AS customer_id, 100.0 AS total_amount
        UNION ALL
        SELECT '2024-07-01', 1, 250.0
        UNION ALL
        SELECT '2025-01-10', 1, 300.0
    ) f
    JOIN DimCustomer_SCD2 c 
        ON f.customer_id = c.customer_id
        AND c.effective_date <= f.order_date
        AND (c.end_date IS NULL OR c.end_date > f.order_date)
'''
display(pd.read_sql(query, conn))

---
## 3. SCD Type 3: Add new column

Зберігає тільки попереднє значення.

In [ ]:
conn.executescript('''
    CREATE TABLE DimCustomer_T3 (
        customer_id INT PRIMARY KEY,
        name TEXT,
        current_city TEXT,
        previous_city TEXT,
        original_city TEXT
    );

    INSERT INTO DimCustomer_T3 VALUES (1, 'Alice', 'Lviv', NULL, 'Lviv');
    INSERT INTO DimCustomer_T3 VALUES (2, 'Bob', 'Kyiv', NULL, 'Kyiv');
''')

print('Before:')
display(pd.read_sql('SELECT * FROM DimCustomer_T3', conn))

# Alice переїжджає
conn.execute('''
    UPDATE DimCustomer_T3 
    SET previous_city = current_city, current_city = 'Odesa'
    WHERE customer_id = 1
''')

print('After move to Odesa:')
display(pd.read_sql('SELECT * FROM DimCustomer_T3', conn))

# Ще один переїзд — original_city зберігає найперше значення
conn.execute('''
    UPDATE DimCustomer_T3 
    SET previous_city = current_city, current_city = 'Kyiv'
    WHERE customer_id = 1
''')

print('After move to Kyiv (тільки prev + original):')
display(pd.read_sql('SELECT * FROM DimCustomer_T3', conn))

---
## 4. SCD Type 4: History table

Основний запис Type 1 (overwrite), історія в окремій таблиці.

In [ ]:
conn.executescript('''
    CREATE TABLE DimCustomer_T4 (
        customer_id INT PRIMARY KEY,
        name TEXT,
        current_city TEXT
    );

    CREATE TABLE DimCustomer_T4_History (
        history_id INTEGER PRIMARY KEY AUTOINCREMENT,
        customer_id INT,
        old_city TEXT,
        changed_from DATE,
        changed_to DATE,
        change_reason TEXT
    );

    INSERT INTO DimCustomer_T4 VALUES (1, 'Alice', 'Lviv');
''')

# Alice переїжджає — оновлюємо main + пишемо history
old_city = conn.execute("SELECT current_city FROM DimCustomer_T4 WHERE customer_id = 1").fetchone()[0]
conn.execute("UPDATE DimCustomer_T4 SET current_city = 'Dnipro' WHERE customer_id = 1")
conn.execute('''
    INSERT INTO DimCustomer_T4_History (customer_id, old_city, changed_from, changed_to, change_reason)
    VALUES (1, ?, '2022-01-01', '2024-06-01', 'Address update')
''', (old_city,))

print('Main table (current):')
display(pd.read_sql('SELECT * FROM DimCustomer_T4', conn))
print('\nHistory table:')
display(pd.read_sql('SELECT * FROM DimCustomer_T4_History', conn))

---
## 5. Автоматизація SCD Type 2 (SQL-процедура)

Реальний SQL для щоденного ETL.

In [ ]:
conn.executescript('''
    CREATE TABLE DimCustomer_PROD (
        customer_key INTEGER PRIMARY KEY AUTOINCREMENT,
        customer_id INT,
        name TEXT,
        city TEXT,
        effective_date DATE,
        end_date DATE,
        is_current BOOLEAN DEFAULT 1
    );

    -- Source даних (OLTP)
    CREATE TABLE src_customer (
        customer_id INT PRIMARY KEY,
        name TEXT,
        city TEXT,
        updated_at DATE
    );

    INSERT INTO src_customer VALUES (1, 'Alice', 'Lviv', '2022-01-01');
    INSERT INTO src_customer VALUES (2, 'Bob', 'Kyiv', '2023-03-15');
''')

# Initial load в DWH
conn.execute('''
    INSERT INTO DimCustomer_PROD (customer_id, name, city, effective_date, end_date, is_current)
    SELECT customer_id, name, city, updated_at, NULL, 1
    FROM src_customer
''')
print('Initial DWH:')
display(pd.read_sql('SELECT * FROM DimCustomer_PROD', conn))

In [ ]:
# Наступний день — Alice змінила місто
conn.execute('''
    UPDATE src_customer SET city = 'Kyiv', updated_at = '2024-06-15' WHERE customer_id = 1
''')

# SCD Type 2 UPSERT (MERGE аналог для SQLite)
# 1. Закриваємо змінені
conn.execute('''
    UPDATE DimCustomer_PROD
    SET end_date = (
        SELECT s.updated_at FROM src_customer s 
        WHERE s.customer_id = DimCustomer_PROD.customer_id
          AND s.city != DimCustomer_PROD.city
    ),
    is_current = 0
    WHERE is_current = 1
      AND customer_id IN (
          SELECT customer_id FROM src_customer s
          JOIN DimCustomer_PROD d ON s.customer_id = d.customer_id
          WHERE d.is_current = 1 AND s.city != d.city
      )
''')

# 2. Додаємо нові версії
conn.execute('''
    INSERT INTO DimCustomer_PROD (customer_id, name, city, effective_date, end_date, is_current)
    SELECT s.customer_id, s.name, s.city, s.updated_at, NULL, 1
    FROM src_customer s
    WHERE s.customer_id IN (
        SELECT customer_id FROM DimCustomer_PROD WHERE is_current = 0 AND customer_id NOT IN (
            SELECT customer_id FROM DimCustomer_PROD WHERE is_current = 1
        )
    )
''')

print('After Alice moved to Kyiv (SCD2 auto):')
display(pd.read_sql('SELECT * FROM DimCustomer_PROD ORDER BY customer_id, effective_date', conn))

---
## 6. Практичні завдання

1. Реалізуйте **SCD Type 2** для таблиці `DimProduct` (ціна продукту змінюється)
2. Додайте **SCD Type 3** колонку `previous_price` в DimProduct
3. Змоделюйте **3 зміни** для клієнта (city, loyalty_tier, name) — кожна через Type 2
4. Напишіть **SQL-запит**, який покаже актуальний стан всіх клієнтів на задану дату
5. (Бонус) Реалізуйте **SCD Type 6** (гібрид): в одній таблиці поточне значення + історія

In [ ]:
conn.close()
print('Connection closed')